<h2>Load and Inspect</h2>


In [32]:
%pip install openpyxl
import pandas as pd


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\JULONG\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [42]:
DATA = "data/pull.xlsx"

df = pd.read_excel(DATA, sheet_name="SB1", header=None, names=["Date", "Price"])
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(by='Date').reset_index(drop=True)

print(df.head())
print(df.tail())


        Date  Price
0 2016-01-01  15.21
1 2016-01-04  14.98
2 2016-01-05  14.60
3 2016-01-06  14.42
4 2016-01-07  14.71
           Date  Price
2705 2026-05-15  14.78
2706 2026-05-18  14.71
2707 2026-05-19  15.04
2708 2026-05-20  14.76
2709 2026-05-21  14.84


<h2>Log Returns</h2>

In [59]:
import numpy as np 

# df["Log_Return"] = np.log(df["Price"]).diff()
# df["Log_Return"] = np.log(df["Price"] / df["Price"].shift(1))
log_ret = df["Log_Return"] = np.log(1 + df["Price"].pct_change().rename("Log_Return"))

print(df.head())

        Date  Price  Log_Return  Roll_Vol
0 2016-01-01  15.21         NaN       NaN
1 2016-01-04  14.98   -0.015237       NaN
2 2016-01-05  14.60   -0.025694       NaN
3 2016-01-06  14.42   -0.012405       NaN
4 2016-01-07  14.71    0.019911       NaN


<h2>Rolling Volatility</h2>

In [62]:
roll_vol = df["Roll_Vol"] = df["Log_Return"].rolling(21).std() * np.sqrt(252)


print(roll_vol)
nan_count = df['Roll_Vol'].isna().sum()
print(f"Number of NaNs in Roll_Vol: {nan_count}")




0            NaN
1            NaN
2            NaN
3            NaN
4            NaN
          ...   
2705    0.300978
2706    0.286778
2707    0.293075
2708    0.304069
2709    0.303848
Name: Log_Return, Length: 2710, dtype: float64
Number of NaNs in Roll_Vol: 21


In [63]:
log_ret

0            NaN
1      -0.015237
2      -0.025694
3      -0.012405
4       0.019911
          ...   
2705   -0.014775
2706   -0.004747
2707    0.022186
2708   -0.018792
2709    0.005405
Name: Log_Return, Length: 2710, dtype: float64

<h2>Chart</h2>

In [69]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Initialize a 2-row subplot layout
fig = make_subplots(
    rows=2, 
    cols=1, 
    shared_xaxes=True, 
    vertical_spacing=0.06,
    subplot_titles=("Sugar Price (c/lb)", "21-Day Rolling Volatility (Annualized)")
)

# 2. Add Price line to Row 1
fig.add_trace(
    go.Scatter(
        x=df["Date"], 
        y=df["Price"], 
        name="Price", 
        line=dict(color="royalblue", width=1.5)
    ),
    row=1, col=1
)

# 3. Add Volatility line to Row 2
fig.add_trace(
    go.Scatter(
        x=df["Date"], 
        y=df["Roll_Vol"], 
        name="Volatility", 
        line=dict(color="darkorange", width=1.5)
    ),
    row=2, col=1
)

# 4. Update layout and axis styling
fig.update_layout(
    title={
        'text': "Sugar (SB1) Price & Historical Volatility",
        'y':0.95,
        'x':0.5,
        'xanchor': 'center',
        'yanchor': 'top'
    },
    height=650,
    template="plotly_white",
    showlegend=False
)

# Add axis titles
fig.update_yaxes(title_text="Price", row=1, col=1)
fig.update_yaxes(title_text="Vol (Annualized)", row=2, col=1)
fig.update_xaxes(title_text="Date", row=2, col=1)

# 5. Display the figure
fig.show()

